# Quais fatores influenciam mais fortemente o desempenho acadêmico dos estudantes e como as instituições educacionais podem atuar para melhorar esses resultados?

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_csv('/content/drive/MyDrive/dados/StudentsPerformance.csv')

df.head()

df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


### Médias de notas por gênero

In [14]:
df.groupby('gender')[['math score','reading score','writing score']].mean()

,math score,reading score,writing score
gender,,,
female,63.633205,72.608108,72.467181
male,68.728216,65.473029,63.311203


In [15]:
# Tradução dos gêneros
traducao_genero = {
    "male": "Masculino",
    "female": "Feminino"
}

df_genero = df.copy()
df_genero['gender'] = df_genero['gender'].map(traducao_genero)

# Cálculo das médias por gênero
gender_means = df_genero.groupby('gender')[['math score', 'reading score', 'writing score']].mean().reset_index()

# Transformar para formato longo
gender_long = gender_means.melt(
    id_vars='gender',
    value_vars=['math score', 'reading score', 'writing score'],
    var_name='Disciplina',
    value_name='Média'
)

# Tradução das disciplinas
trad_disc = {
    'math score': 'Matemática',
    'reading score': 'Leitura',
    'writing score': 'Escrita'
}
gender_long['Disciplina'] = gender_long['Disciplina'].map(trad_disc)

# Gráfico
fig = px.bar(
    gender_long,
    x='gender',
    y='Média',
    color='Disciplina',
    barmode='group',
    title='Médias das notas por gênero',
    labels={'gender': 'Gênero'}
)

fig.update_layout(template='plotly_white')

fig.show()

In [16]:
# Tradução dos gêneros
traducao_genero = {
    "male": "Masculino",
    "female": "Feminino"
}

df_plot = df.rename(columns={
    'math score': 'Matemática',
    'reading score': 'Leitura',
    'writing score': 'Escrita'
}).copy()

df_plot['gender'] = df_plot['gender'].map(traducao_genero)

# Converter para formato longo
df_long = df_plot.melt(
    id_vars='gender',
    value_vars=['Matemática', 'Leitura', 'Escrita'],
    var_name='Disciplina',
    value_name='Nota'
)

# Histograma por gênero
fig = px.histogram(
    df_long,
    x='Nota',
    color='gender',
    facet_row='Disciplina',     # um painel por disciplina
    barmode='overlay',
    opacity=0.6,
    title='Distribuição das notas por gênero',
    labels={'gender': 'Gênero'},
    template='plotly_white'
)

fig.update_layout(
    height=900,
)

fig.show()

**Matemática**

Homens apresentam ligeira vantagem na mediana.

A dispersão é semelhante entre gêneros, sugerindo que diferenças não são extremas.

**Leitura**

Distribuição feminina é deslocada para notas mais altas de forma consistente.

Homens têm maior variação inferior, indicando mais estudantes com dificuldades em leitura.

**Escrita**

Mulheres apresentam desempenho superior também nesta área.

A escrita é a disciplina com maior diferença de gênero.

### Impacto do curso preparatório

In [17]:
df.groupby('test preparation course')[['math score','reading score','writing score']].mean()

,math score,reading score,writing score
test preparation course,,,
completed,69.695531,73.893855,74.418994
none,64.077882,66.534268,64.504673


In [18]:
# Tradução das categorias do curso preparatório
traducao_curso = {
    "completed": "Concluído",
    "none": "Não fez"
}

df_plot = df.copy()
df_plot['test preparation course'] = df_plot['test preparation course'].map(traducao_curso)

subjects = {
    'math score': 'Matemática',
    'reading score': 'Leitura',
    'writing score': 'Escrita'
}

figs = []

for col, title in subjects.items():
    fig = px.box(
        df_plot,
        x='test preparation course',
        y=col,
        points="outliers",
        title=f"Impacto do curso preparatório em {title}",
        labels={
            'test preparation course': 'Curso preparatório',
            col: 'Nota'
        },
        template="plotly_white"
    )
    figs.append(fig)

# Mostrar as três figuras separadamente
for fig in figs:
    fig.show()

Os dados mostram que o curso preparatório exerce o maior impacto positivo sobre o desempenho dos estudantes entre todas as variáveis analisadas. Estudantes que concluíram o curso apresentam, em média, 10 a 15 pontos a mais em Matemática, Leitura e Escrita.

Além de elevar as médias, o curso reduz a dispersão das notas, diminuindo a quantidade de desempenhos muito baixos. Isso indica que a intervenção não só melhora resultados, mas também mitiga desigualdades internas.

O efeito é particularmente forte em Leitura e Escrita, sugerindo que o curso favorece sobretudo habilidades relacionadas à linguagem.

### Impacto do tipo de almoço (proxy de renda)

In [19]:
df.groupby('lunch')[['math score','reading score','writing score']].mean()

,math score,reading score,writing score
lunch,,,
free/reduced,58.921127,64.653521,63.022535
standard,70.034109,71.654264,70.823256


In [20]:
# Tradução dos tipos de almoço
traducao_lunch = {
    "standard": "Padrão",
    "free/reduced": "Gratuito/Reduzido"
}

df_plot = df.copy()
df_plot['lunch'] = df_plot['lunch'].map(traducao_lunch)

# Cálculo das médias
lunch_means = df_plot.groupby('lunch')[['math score','reading score','writing score']].mean()

fig = go.Figure(data=[
    go.Bar(name='Matemática', x=lunch_means.index, y=lunch_means['math score']),
    go.Bar(name='Leitura',    x=lunch_means.index, y=lunch_means['reading score']),
    go.Bar(name='Escrita',    x=lunch_means.index, y=lunch_means['writing score'])
])

fig.update_layout(
    barmode='group',
    title='Médias das notas por tipo de almoço',
    xaxis_title='Tipo de almoço',
    yaxis_title='Média da nota',
    template='plotly_white'
)

fig.show()

Os estudantes que recebem almoço gratuito ou reduzido, indicador de menor condição socioeconômica, apresentam médias de nota mais baixas em todas as disciplinas quando comparados aos alunos com almoço padrão.

O tipo de almoço — e, por consequência, a condição socioeconômica — é um dos principais fatores associados ao desempenho acadêmico.

### Escolaridade dos pais

In [21]:
df.groupby('parental level of education')[['math score','reading score','writing score']].mean().sort_values('math score')

,math score,reading score,writing score
parental level of education,,,
high school,62.137755,64.704082,62.448980
some high school,63.497207,66.938547,64.888268
some college,67.128319,69.460177,68.840708
associate's degree,67.882883,70.927928,69.896396
bachelor's degree,69.389831,73.000000,73.381356
master's degree,69.745763,75.372881,75.677966


In [22]:
from plotly.subplots import make_subplots

# Tradução dos níveis de escolaridade
traducao_escolaridade = {
    "some high school": "Algum ensino médio",
    "high school": "Ensino médio",
    "some college": "Alguma faculdade",
    "associate's degree": "Tecnólogo",
    "bachelor's degree": "Bacharelado",
    "master's degree": "Mestrado"
}

df_plot = df.copy()
df_plot['parental level of education'] = (
    df_plot['parental level of education'].map(traducao_escolaridade)
)

# Disciplinas e títulos
subjects = {
    'math score': 'Média de Matemática por escolaridade dos pais',
    'reading score': 'Média de Leitura por escolaridade dos pais',
    'writing score': 'Média de Escrita por escolaridade dos pais'
}

# Criar figure com subplots
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=list(subjects.values())
)

# Adicionar barras
for i, col in enumerate(subjects.keys(), start=1):

    medias = df_plot.groupby('parental level of education')[col].mean().reset_index()

    fig.add_trace(
        go.Bar(
            x=medias['parental level of education'],
            y=medias[col],
        ),
        row=i, col=1
    )

    fig.update_yaxes(title_text="Média da nota", row=i, col=1)

# Ajustes gerais
fig.update_layout(
    height=1100,
    width=900,
    template="plotly_white",
    title="Médias das notas por escolaridade dos pais",
    showlegend=False
)

fig.update_xaxes(tickangle=45)

fig.show()

As médias das notas mostram que o desempenho dos estudantes aumenta conforme cresce o nível de escolaridade dos pais. Alunos cujos responsáveis possuem bacharelado ou mestrado apresentam resultados consistentemente superiores em Matemática, Leitura e Escrita.

### Correlação entre as notas

In [23]:
notas = df[['math score', 'reading score', 'writing score']]
corr = notas.corr()
fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="Blues",
    title="Correlação entre as notas"
)

fig.update_layout(
    width=600,
    height=400
)

fig.show()

A análise da matriz de correlação revela que Leitura e Escrita possuem a relação mais forte, com correlação muito alta, indicando que estudantes que têm bom desempenho em leitura tendem quase sempre a apresentar bom desempenho em escrita.

Matemática apresenta correlações moderadas com Leitura e Escrita, sugerindo que o desempenho matemático está relacionado às habilidades de linguagem, mas de forma menos intensa.

### Distribuição das notas

In [24]:
# Colunas das notas e tradução
df_plot = df.rename(columns={
    'math score': 'Matemática',
    'reading score': 'Leitura',
    'writing score': 'Escrita'
})

# Converter para formato longo (necessário para px.histogram)
df_long = df_plot.melt(
    value_vars=['Matemática', 'Leitura', 'Escrita'],
    var_name='Disciplina',
    value_name='Nota'
)

# Histograma
fig = px.histogram(
    df_long,
    x='Nota',
    color='Disciplina',
    opacity=0.6,
    barmode='overlay',
    title='Distribuição Geral das Notas',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title="Nota",
    yaxis_title="Frequência"
)

fig.show()

Matemática

In [25]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribuição das notas de Matemática",
                    "Boxplot de Matemática por gênero")
)

# Histograma
fig.add_trace(
    go.Histogram(
        x=df['math score'],
        opacity=0.75
    ),
    row=1, col=1
)

# Boxplot
fig.add_trace(
    go.Box(
        x=df['gender'],
        y=df['math score'],
        boxmean=True
    ),
    row=1, col=2
)

fig.update_layout(
    width=950, height=400,
    template="plotly_white",
    showlegend=False
)

fig.update_xaxes(title_text="Nota", row=1, col=1)
fig.update_yaxes(title_text="Frequência", row=1, col=1)

fig.update_xaxes(title_text="Gênero", row=1, col=2)
fig.update_yaxes(title_text="Nota", row=1, col=2)

fig.show()

Leitura

In [26]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribuição das notas de Leitura",
                    "Boxplot de Leitura por gênero")
)

# Histograma
fig.add_trace(
    go.Histogram(
        x=df['reading score'],
        opacity=0.75
    ),
    row=1, col=1
)

# Boxplot
fig.add_trace(
    go.Box(
        x=df['gender'],
        y=df['reading score'],
        boxmean=True
    ),
    row=1, col=2
)

fig.update_layout(
    width=950, height=400,
    template="plotly_white",
    showlegend=False
)

fig.update_xaxes(title_text="Nota", row=1, col=1)
fig.update_yaxes(title_text="Frequência", row=1, col=1)

fig.update_xaxes(title_text="Gênero", row=1, col=2)
fig.update_yaxes(title_text="Nota", row=1, col=2)

fig.show()


Escrita

In [27]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribuição das notas de Escrita",
                    "Boxplot de Escrita por gênero")
)

# Histograma
fig.add_trace(
    go.Histogram(
        x=df['writing score'],
        opacity=0.75
    ),
    row=1, col=1
)

# Boxplot
fig.add_trace(
    go.Box(
        x=df['gender'],
        y=df['writing score'],
        boxmean=True
    ),
    row=1, col=2
)

fig.update_layout(
    width=950, height=400,
    template="plotly_white",
    showlegend=False
)

fig.update_xaxes(title_text="Nota", row=1, col=1)
fig.update_yaxes(title_text="Frequência", row=1, col=1)

fig.update_xaxes(title_text="Gênero", row=1, col=2)
fig.update_yaxes(title_text="Nota", row=1, col=2)

fig.show()


**Matemática**

Há uma quantidade relevante de estudantes com notas entre 50 e 70, sugerindo dificuldade moderada na disciplina.

O pico da distribuição ocorre abaixo de 80, indicando que altas performances são menos frequentes.

**Leitura**

Distribuição mais concentrada à direita, com muitas notas acima de 70.

Desempenho significativamente melhor que matemática, com menos valores baixos.

Indica que leitura é a área mais forte da maioria dos estudantes.

**Escrita**

Muito semelhante à leitura, com concentração de notas altas.

Leve cauda esquerda indica que alguns estudantes têm dificuldades específicas em escrita.

A proximidade entre leitura e escrita reflete a correlação alta entre as duas competências.

### Conclusão

Em conjunto, os dados evidenciam que condições socioeconômicas, acesso a preparação e apoio educacional influenciam significativamente o desempenho escolar, revelando desigualdades estruturais e oportunidades claras para intervenções pedagógicas.

Algumas ações possíveis:

1) Criar ou subsidiar cursos preparatórios
2) Oferecer apoio adicional a estudantes em situação de vulnerabilidade socioeconômica
3) Implementar ações de suporte acadêmico, especialmente em matemática
4) Desenvolver iniciativas que apoiem o desenvolvimento de competências de linguagen